In [ ]:
# ESP32 PDF RAG Project
# This notebook loads an ESP32 PDF, splits the text into chunks, creates embeddings, stores them in FAISS, and enables similarity search from PDF content.

In [ ]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

pdf_path = Path("esp32_datasheet_en.pdf")
if not pdf_path.exists():
    raise FileNotFoundError(f"PDF not found: {pdf_path}. Place the file in the notebook folder or update the path.")

loader = PyPDFLoader(str(pdf_path))
documents = loader.load()
print(f"Loaded {len(documents)} document(s) from {pdf_path}")
print(documents[0].page_content[:500])

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

docs = text_splitter.split_documents(documents)
print("Created chunks:", len(docs))
print(docs[0].page_content[:500])

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
db = FAISS.from_documents(docs, embeddings)
print("FAISS vector store created with:", len(docs), "documents")

In [ ]:
query = input("Enter your ESP32 question: " )
results = db.similarity_search(query)
for i, result in enumerate(results):
    print("--- Result #", i + 1, "---")
    print(result.page_content[:1000])